# 38. Prefill/Decode Scheduling | Prefill/Decode 调度
**难度：** Medium | **环境：** CPU-first | **标签：** `推理优化`, `Serving`, `Chunked Prefill`, `PD Disaggregation` | **目标人群：** 推理优化学习者

---

## 本节导读

长 prompt 到达时，完整 Prefill 可能占住一整轮批次，使正在生成的请求等待更久。Chunked Prefill 先把未处理输入切成可让出的执行块；当 Prefill 与 Decode 的资源压力长期不同，再考虑把它们组织成独立服务池。

本节建立“请求压力 → 分块推进 → 分池与交接 → 服务决策”的链路。它先解释调度如何减少请求间的相互阻塞，再判断 PD 分离的吞吐和延迟收益能否覆盖交接成本。

**关键词：** `chunked prefill`, `continuous batching`, `PD disaggregation`, `handoff`

## 前置阅读

**导语：** 先能区分 Prefill 的突发输入处理与 Decode 的持续 KV 读取，再观察未命中 suffix 为什么要按可让出的粒度进入批次。

- [34. Prefix Cache Matching and Reuse | Prefix Cache 匹配与复用](./34_Prefix_Cache_Matching_and_Reuse.ipynb)
- [36. Decode Scheduling | Decode 调度](./36_Decode_Scheduling.ipynb)
- [37. KV Cache Scheduling | KV Cache 调度](./37_KV_Cache_Scheduling.ipynb)

### Step 1: 请求为什么需要同时考虑 Prefill 与 Decode

一次请求先处理完整输入，再逐 token 生成输出。长 prompt 往往带来集中的 Prefill 计算和访存；长生成则长期占用 KV Cache，并对 Decode 等待敏感。Continuous Batching 在每一轮重新查看 ready 请求：正在等待下一个 token 的 Decode 与可继续处理的 Prefill suffix 都进入同一次准入判断。

| 请求形态 | 更突出的压力 | 首先观察什么 |
|---|---|---|
| 长输入、短生成 | Prefill 计算、输入访存与批次占用 | TTFT、Prefill 队列、Decode 抖动 |
| 短输入、长生成 | KV Cache 驻留、逐步 Decode 与排队 | TPOT、KV 使用量、Decode 队列 |
| 输入和生成都长 | 两类压力叠加 | E2E、P95/P99、池利用率 |

![Continuous Batching：Chunked Prefill 与 Decode 的调度时间线](../docs/public/02_PyTorch_Algorithms/38_prefill_decode_timeline.svg)

### Step 2: Chunked Prefill 如何保留 Decode 的准入机会

Prefix Cache 未命中的 suffix 仍要 Prefill。Chunked Prefill 将它切成有限大小的块：每完成一个 chunk，调度器便回到 ready 集合，决定继续当前请求还是先服务等待中的 Decode。分块减少长输入连续占用 batch 的时间；prefix 是否命中仍由缓存机制决定。

| 调度对象 | 含义 | 影响 |
|---|---|---|
| `suffix` | 本次仍需处理的未命中输入 | 决定新增 Prefill 工作量 |
| `chunk_size` | 单次进入 Prefill 的最大 token 数 | 控制单轮占用和让出频率 |
| `chunk_plan` | suffix 的有序分块计划 | 让调度器在块边界重新选请求 |
| `yield_point` | 一个 chunk 完成后的可重排位置 | 缓解长 Prefill 对 Decode 的阻塞 |

![Chunked Prefill：后缀分块与可让出执行计划](../docs/public/02_PyTorch_Algorithms/38_chunked_prefill_schedule.svg)

### Step 3: 何时采用 PD 分池与状态交接

若 Chunked Prefill 后两类请求仍长期争用同一资源池，可以把 Prefill 与 Decode 分配给不同的逻辑池。此时需要把交接时间、链路传输、池利用率和尾延迟与吞吐一起比较，才能判断分池是否适合当前 workload。

| 决策层次 | 主要动作 | 需要同时记录 |
|---|---|---|
| 同池分块调度 | 在 chunk 边界重排 Prefill / Decode | TTFT、TPOT、Decode 抖动 |
| 逻辑 PD 分池 | 将请求分到 prefill / decode / shared 池 | 吞吐、P95、池利用率 |
| 跨池交接 | Prefill 结束后交接状态给 Decode 池 | 交接时间、传输量、失败状态 |
| 保留或调整 | 比较共享池与拆分方案 | 吞吐、尾延迟与交接代价共同达标 |

![Prefill 与 Decode 的拆分流程：分类、交接与指标比较](../docs/public/02_PyTorch_Algorithms/38_pd_disaggregation.svg)

### Step 4：实现 CPU 调度账本与决策

题目区把本节的四个机制写成可测试函数：先识别请求压力，再构造可让出的 Prefill chunk，随后建立逻辑 PD 分池，最后比较共享池与分池的吞吐、P95 和交接预算。

| TODO | 机制责任 | 测试关注点 |
|:---|:---|:---|
| TODO 1 | 将请求分为 prefill-heavy、decode-heavy 与 mixed | 分类数量覆盖全部请求 |
| TODO 2 | 保持顺序地切分未命中 suffix | 尾块与非法 chunk size |
| TODO 3 | 让每个请求只进入一个逻辑池 | 请求不遗漏、不重复 |
| TODO 4 | 用吞吐、P95 和 handoff 预算判断是否保留 PD | 不能用单一指标决定 |


In [ ]:
from typing import Dict, List


In [ ]:
def summarize_request_mix(requests: List[Dict[str, int]], long_prompt_threshold: int) -> Dict[str, int]:
    """统计 prefill-heavy、decode-heavy 与 mixed 请求数量。"""
    # TODO 1（请求压力）：根据 prompt_tokens 与 decode_tokens 返回完整三类计数字典。
    # result = ???
    raise NotImplementedError


def build_chunked_prefill_plan(suffix_tokens: List[int], chunk_size: int) -> List[List[int]]:
    """把未命中 suffix 切成有序 chunk；尾块可以小于 chunk_size。"""
    # TODO 2（分块计划）：按 chunk_size 保持顺序切分 suffix；尾块可以更短。
    # chunks = ???
    raise NotImplementedError


def plan_pd_split(requests: List[Dict[str, int]], long_prompt_threshold: int, long_decode_threshold: int) -> Dict[str, List[str]]:
    """按请求压力生成 prefill、decode 与 shared 三个逻辑池。"""
    # TODO 3（逻辑分池）：每个请求恰好进入 prefill_pool、decode_pool 或 shared_pool。
    # pools = ???
    raise NotImplementedError


def evaluate_pd_decision(baseline: Dict[str, float], split_run: Dict[str, float], max_handoff_ms: float) -> Dict[str, object]:
    """比较共享池与 PD 分池，并同时检查吞吐、P95 和交接预算。"""
    # TODO 4（分池决策）：吞吐提升、P95 不恶化且 handoff 在预算内时才保留拆分。
    # decision = ???
    raise NotImplementedError

In [ ]:
# 机制测试：分别检查请求分类、suffix 分块、逻辑分池和交接预算。
REQUESTS = [
    {'name': 'a', 'prompt_tokens': 4000, 'decode_tokens': 64},
    {'name': 'b', 'prompt_tokens': 256, 'decode_tokens': 512},
    {'name': 'c', 'prompt_tokens': 1500, 'decode_tokens': 128},
]


def test_request_mix_classification():
    """验证三类请求统计覆盖全部输入。"""
    summary = summarize_request_mix(REQUESTS, long_prompt_threshold=2048)
    assert summary == {'prefill_heavy': 1, 'decode_heavy': 1, 'mixed': 1}
    assert sum(summary.values()) == len(REQUESTS), '请求分类数量必须守恒'


def test_chunked_prefill_plan():
    """验证 suffix 顺序、尾块和非法 chunk_size。"""
    assert build_chunked_prefill_plan([9, 10, 11, 12, 13], chunk_size=2) == [[9, 10], [11, 12], [13]]
    try:
        build_chunked_prefill_plan([1], chunk_size=0)
    except ValueError:
        pass
    else:
        raise AssertionError('非正 chunk_size 必须拒绝')


def test_pd_pool_assignment():
    """验证每个请求只进入一个逻辑池。"""
    plan = plan_pd_split(REQUESTS, long_prompt_threshold=2048, long_decode_threshold=256)
    assert plan == {'prefill_pool': ['a'], 'decode_pool': ['b'], 'shared_pool': ['c']}


def test_pd_decision_and_handoff_budget():
    """验证吞吐、P95 与 handoff 预算共同决定是否保留拆分。"""
    baseline = {'throughput': 100, 'p95_latency_ms': 180}
    candidate = {'throughput': 126, 'p95_latency_ms': 150, 'handoff_ms': 8}
    assert evaluate_pd_decision(baseline, candidate, max_handoff_ms=10)['keep_split'] is True
    reject = evaluate_pd_decision(baseline, {**candidate, 'handoff_ms': 18}, max_handoff_ms=10)
    assert reject['keep_split'] is False, '交接超过预算时不能直接保留 PD'


def test_prefill_decode_scheduling():
    """汇总四组调度机制测试。"""
    try:
        test_request_mix_classification()
        test_chunked_prefill_plan()
        test_pd_pool_assignment()
        test_pd_decision_and_handoff_budget()
        print('✅ Prefill/Decode 调度机制测试通过：分类、分块、分池和决策均通过。')
    except NotImplementedError:
        raise
    except (NameError, AttributeError, TypeError, ValueError, AssertionError) as error:
        raise NotImplementedError('请先完成 TODO 代码或检查字段名！') from error


test_prefill_decode_scheduling()


## 参考代码与解析

### 代码


In [ ]:
def summarize_request_mix(requests: List[Dict[str, int]], long_prompt_threshold: int) -> Dict[str, int]:
    """统计 prefill-heavy、decode-heavy 与 mixed 请求数量。"""
    # TODO 1: 按请求的输入和生成长度完成三类统计
    result = {'prefill_heavy': 0, 'decode_heavy': 0, 'mixed': 0}
    for request in requests:
        prompt_tokens = request.get('prompt_tokens', 0)
        decode_tokens = request.get('decode_tokens', 0)
        if prompt_tokens > long_prompt_threshold and decode_tokens <= long_prompt_threshold // 8:
            result['prefill_heavy'] += 1
        elif decode_tokens > long_prompt_threshold // 8 and prompt_tokens <= long_prompt_threshold:
            result['decode_heavy'] += 1
        else:
            result['mixed'] += 1
    return result


def build_chunked_prefill_plan(suffix_tokens: List[int], chunk_size: int) -> List[List[int]]:
    """把未命中 suffix 切成有序 chunk；尾块可以小于 chunk_size。"""
    # TODO 2: 生成 Chunked Prefill 的执行计划
    if chunk_size <= 0:
        raise ValueError('chunk_size 必须为正数')
    suffix = list(suffix_tokens)
    return [suffix[start:start + chunk_size] for start in range(0, len(suffix), chunk_size)]


def plan_pd_split(requests: List[Dict[str, int]], long_prompt_threshold: int, long_decode_threshold: int) -> Dict[str, List[str]]:
    """按请求压力生成 prefill、decode 与 shared 三个逻辑池。"""
    # TODO 3: 每个请求只进入一个逻辑池
    prefill_pool, decode_pool, shared_pool = [], [], []
    for request in requests:
        name = request.get('name', 'request')
        if request.get('prompt_tokens', 0) > long_prompt_threshold and request.get('decode_tokens', 0) <= long_decode_threshold:
            prefill_pool.append(name)
        elif request.get('decode_tokens', 0) > long_decode_threshold and request.get('prompt_tokens', 0) <= long_prompt_threshold:
            decode_pool.append(name)
        else:
            shared_pool.append(name)
    return {'prefill_pool': prefill_pool, 'decode_pool': decode_pool, 'shared_pool': shared_pool}


def evaluate_pd_decision(baseline: Dict[str, float], split_run: Dict[str, float], max_handoff_ms: float) -> Dict[str, object]:
    """比较共享池与 PD 分池，并同时检查吞吐、P95 和交接预算。"""
    # TODO 4: 输出是否保留当前 PD 方案
    throughput_gain = split_run.get('throughput', 0.0) - baseline.get('throughput', 0.0)
    latency_delta_ms = split_run.get('p95_latency_ms', 0.0) - baseline.get('p95_latency_ms', 0.0)
    handoff_ms = split_run.get('handoff_ms', 0.0)
    handoff_ok = handoff_ms <= max_handoff_ms
    keep_split = throughput_gain > 0 and latency_delta_ms <= 0 and handoff_ok
    return {'throughput_gain': throughput_gain, 'latency_delta_ms': latency_delta_ms, 'handoff_ms': handoff_ms, 'keep_split': keep_split}

### 解析

**TODO 1：识别请求压力**

- 统计结果必须覆盖全部请求；它把输入长度和生成长度变成调度器可以使用的请求画像。

**TODO 2：生成 Chunked Prefill 计划**

- 只切分 Prefix Cache 未命中的 suffix，不改变 token 顺序；每个 chunk 结束后，调度器才有机会重新安排 Decode 与后续 Prefill。

**TODO 3：建立逻辑 PD 分池**

- 每个请求只进入 prefill、decode 或 shared 之一。这个账本帮助检查“谁在争用什么资源”，也避免重复服务。

**TODO 4：判断是否保留分池**

- 分池必须同时带来吞吐改善、P95 不恶化，并让 handoff 保持在预算内；任何一项失败，都应继续调整而不是直接保留。

### Step 5: 可选 GPU / backend：验证调度、分池与交接代价

#### 5.1 环境、输入与固定条件

CPU 题目区验证请求分类、suffix 分块、逻辑分池和决策规则。GPU/backend 实验先固定模型、dtype、prompt/decode 分布、并发度和 chunk size；单 GPU smoke test 只建立基线，不能代替跨池交接结论。


| 实验路径 | 使用资产 | 学习者操作 | 可以验证 / 不能直接推出 |
| --- | --- | --- | --- |
| CPU 机制验证 | 题目区四个函数和测试单元 | 检查分块、分池、P95 与交接预算判断 | 机制账本；不能推出真实 worker 性能 |
| 环境预检 | `tools/environment_preflight.py`、当前 Notebook runtime | 检查 CUDA、GPU、显存、PyTorch 与可选 backend | 环境可执行性；不产生 Serving 结论 |
| 单 GPU backend | vLLM / SGLang 与统一请求集 | 建立 shared pool、Chunked Prefill 或单 worker baseline | 请求指标；不能证明跨设备交接 |
| 多 GPU / serving backend | 两类 worker、状态交接和统一 workload | 对比 shared、chunked 与 PD split | TTFT、TPOT、吞吐、P95、交接时间与池利用率 |
| 异构 PD | 能力不同的资源池和链路记录 | 进入 39 的路由、传输/重算与回退判断 | 当前资源组合；不能外推到其他拓扑 |

#### 5.2 配置与执行

真实 backend 验证优先参考 [70 Serving Scheduler Benchmark](./70_Serving_Scheduler_Benchmark.ipynb)、[39 异构 PD 与服务分层](./39_Hetero_PD_and_Serving_Tiers.ipynb) 与 [81 Distributed Inference Project](./81_Distributed_Inference_Project.ipynb)。每组结果必须记录 `evidence_level`；CPU 模拟值不能写成真实 PD 收益。

In [ ]:
"""GPU/backend 配置：先明确 PD 资源形态，默认只做环境预检。"""
RUN_PD_GPU_PREFLIGHT = False
PD_RESULT_PATH = 'benchmarks/results/38_prefill_decode_scheduling.json'
PD_BACKEND = 'vllm'  # vllm / sglang / multi_backend
PD_RESOURCE_MODE = 'homogeneous'  # homogeneous / heterogeneous；异构 PD 的路由逻辑见 39。
PD_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
PD_DTYPE = 'float16'
PD_PROMPT_TOKENS = 2048
PD_DECODE_TOKENS = 256
PD_CONCURRENCY = 2
PD_WORKER_COUNT = 2

In [ ]:
"""GPU/backend 执行单元：记录环境和实验计划，不伪造 backend 结果。"""
if RUN_PD_GPU_PREFLIGHT:
    import importlib.util
    import json
    from pathlib import Path
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('PD 实验预检需要 CUDA；请先切换到 GPU runtime。')
    if PD_BACKEND not in {'vllm', 'sglang', 'multi_backend'}:
        raise ValueError('PD_BACKEND 必须是 vllm、sglang 或 multi_backend。')
    if any(value < 1 for value in (PD_PROMPT_TOKENS, PD_DECODE_TOKENS, PD_CONCURRENCY, PD_WORKER_COUNT)):
        raise ValueError('token 数、并发度和 worker 数必须为正数。')

    project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'benchmarks').is_dir()), Path.cwd())
    report = {
        'task': 'prefill_decode_scheduling_preflight',
        'evidence_level': 'gpu_environment_preflight_only',
        'runtime': {
            'device': torch.cuda.get_device_name(0),
            'gpu_memory_gb': round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2),
            'torch': torch.__version__,
            'torch_cuda': torch.version.cuda,
            'backend_installed': importlib.util.find_spec(PD_BACKEND) is not None if PD_BACKEND != 'multi_backend' else False,
        },
        'config': {'model': PD_MODEL_ID, 'dtype': PD_DTYPE, 'prompt_tokens': PD_PROMPT_TOKENS, 'decode_tokens': PD_DECODE_TOKENS, 'concurrency': PD_CONCURRENCY, 'worker_count': PD_WORKER_COUNT, 'backend': PD_BACKEND, 'resource_mode': PD_RESOURCE_MODE},
        'decision': {'decision': 'measure', 'reason': '环境预检完成；真实 PD split 仍需两个 worker 池和统一 workload。'},
    }
    output_path = project_root / PD_RESULT_PATH
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print('PD GPU/backend 预检未启动：将 RUN_PD_GPU_PREFLIGHT 改为 True 后运行。')

In [ ]:
# 5.3 只读取已保存的预检或 backend 结果；不会启动 worker。
import json
from pathlib import Path

result_path = Path(PD_RESULT_PATH)
if result_path.exists():
    report = json.loads(result_path.read_text(encoding='utf-8'))
    print({key: report.get(key) for key in ('task', 'runtime', 'config', 'evidence_level', 'decision')})
else:
    print(f'尚未找到结果文件：{result_path}。先在 5.2 运行预检或保存真实 backend 结果。')


#### 5.4 记录结果与证据

每组新条件新增一行；没有真实 worker 交接数据时保留为空。记录表用于区分 CPU 账本、环境预检与真实 PD backend 结果。

| 配置组 | GPU / backend | 资源形态 | 模型与 dtype | worker 配置 | chunk size | prompt / decode tokens | 并发 | TTFT / TPOT | 吞吐 | P95 | 交接时间 | 峰值显存 | evidence level | decision | 结果文件 |
| --- | --- | --- | --- | --- | ---: | --- | ---: | --- | ---: | ---: | ---: | ---: | --- | --- | --- |
| 示例：CPU 账本 | CPU / 模拟 | logical pools | 待填写 | shared / split 计划 | 待填写 | 待填写 | 待填写 | 不适用 | 不适用 | 不适用 | 不适用 | 不适用 | cpu_simulation | 待填写 | 待填写 |
|  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |

## 相关阅读

完成 Chunked Prefill、分池与交接预算判断后，可以继续阅读异构 PD、真实服务调度和多实例验证。

- [DistServe 原论文：Disaggregating Prefill and Decoding for Goodput-optimized Large Language Model Serving](https://arxiv.org/abs/2401.09670)
- [vLLM 官方仓库](https://github.com/vllm-project/vllm)
- [SGLang PD Disaggregation 文档](https://docs.sglang.ai/advanced_features/pd_disaggregation.html)
- [39. Hetero PD and Serving Tiers | 异构 PD 与服务分层](./39_Hetero_PD_and_Serving_Tiers.ipynb)
- [70. Serving Scheduler Benchmark | 推理服务调度基准](./70_Serving_Scheduler_Benchmark.ipynb)